# NLP Masterclass 03: Recurrent Networks & LSTMs

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** NLP 02 (Embeddings), DL 01 (Neural Network Mathematics)  
**Difficulty:** ⭐⭐ Intermediate  
**Estimated Time:** 90–120 minutes

---

## 🎓 Socratic Deep Check

> *"RNNs process sequences one token at a time, maintaining a hidden state. Transformers process ALL tokens in parallel. If transformers are faster AND better, why do some production systems still use LSTMs?"*

---

## Why Learn RNNs in the Transformer Era?

1. **Interview prep** — RNN/LSTM questions are still asked at FAANG companies
2. **Understanding attention** — Attention was invented to FIX RNN limitations (NLP 04)
3. **Edge deployment** — LSTMs are tiny. A 10KB LSTM runs on a microcontroller where a 7B transformer cannot
4. **Time-series** — For many non-NLP sequence tasks (sensor data, stock prices), LSTMs remain competitive

## Table of Contents
1. Vanilla RNN: Forward & Backward Through Time
2. The Vanishing Gradient Catastrophe
3. LSTM: Long Short-Term Memory (from scratch)
4. GRU: Gated Recurrent Unit
5. Bidirectional & Stacked Architectures

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors see RNNs and Transformers as unrelated architectures. Seniors understand the EVOLUTIONARY chain: RNN → RNN+Attention → Pure Attention (Transformer). Each step solved a specific limitation of the previous one. Understanding this evolution explains WHY transformers are designed the way they are.

**Mental Model:** An RNN is like reading a book one word at a time and writing a summary after each word. By page 100, your summary has lost details from page 1 — that's vanishing gradients. An LSTM adds a notebook (cell state) where important details persist. A Transformer reads the entire book simultaneously — which is why it's better but more expensive.

**Common Junior Pitfall:** Trying to use an RNN/LSTM for tasks that require long-range dependencies (e.g., document summarization with 5000+ tokens). LSTMs practically fail beyond ~200-500 tokens. Use transformers.

---

In [ ]:
!pip install -q torch numpy matplotlib

## 1 · Vanilla RNN

### The Recurrence Formula

$$h_t = \tanh(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)$$
$$y_t = W_{hy} \cdot h_t + b_y$$

| Component | Shape | Purpose |
|-----------|-------|---------|
| $x_t$ | (input_dim,) | Current input |
| $h_t$ | (hidden_dim,) | Hidden state = "memory" |
| $W_{xh}$ | (input_dim, hidden_dim) | Input-to-hidden |
| $W_{hh}$ | (hidden_dim, hidden_dim) | Hidden-to-hidden (recurrence) |

```
Time:  t=0      t=1      t=2      t=3
Input: "The"    "cat"    "sat"    "down"
         ↓        ↓        ↓        ↓
       [RNN] → [RNN] → [RNN] → [RNN] → h₃ (final state)
         ↓        ↓        ↓        ↓
       h₀       h₁       h₂       h₃
```

In [ ]:
import numpy as np
import torch
import torch.nn as nn

class VanillaRNN:
    """RNN from scratch using NumPy."""
    
    def __init__(self, input_dim, hidden_dim, output_dim):
        scale = 0.01
        self.W_xh = np.random.randn(input_dim, hidden_dim) * scale
        self.W_hh = np.random.randn(hidden_dim, hidden_dim) * scale
        self.W_hy = np.random.randn(hidden_dim, output_dim) * scale
        self.b_h = np.zeros((1, hidden_dim))
        self.b_y = np.zeros((1, output_dim))
        self.hidden_dim = hidden_dim
    
    def forward(self, inputs):
        """Process a sequence step by step.
        
        Args:
            inputs: list of input vectors [(input_dim,), ...]
        Returns:
            outputs: list of output vectors
            hidden_states: list of hidden states
        """
        h = np.zeros((1, self.hidden_dim))  # Initial hidden state
        outputs, hidden_states = [], []
        
        for x_t in inputs:
            x_t = x_t.reshape(1, -1)
            # The recurrence: combine current input with previous memory
            h = np.tanh(x_t @ self.W_xh + h @ self.W_hh + self.b_h)
            y = h @ self.W_hy + self.b_y
            outputs.append(y.squeeze())
            hidden_states.append(h.squeeze())
        
        return outputs, hidden_states

# Demo: Process a 5-word sequence
rnn = VanillaRNN(input_dim=8, hidden_dim=16, output_dim=4)
fake_sentence = [np.random.randn(8) for _ in range(5)]  # 5 tokens, 8-dim embeddings

outputs, hidden_states = rnn.forward(fake_sentence)
print('Vanilla RNN Processing:')
for t, (out, h) in enumerate(zip(outputs, hidden_states)):
    print(f'  Step {t}: hidden={h[:4].round(3)} output={out.round(3)}')
print(f'\nFinal hidden state captures "memory" of the entire sequence')
print(f'This is used for classification, similar to BERT\'s [CLS] token')

---
## 2 · The Vanishing Gradient Catastrophe

### The Problem

During backpropagation through time (BPTT), gradients are multiplied by $W_{hh}$ at every time step:

$$\frac{\partial h_T}{\partial h_1} = \prod_{t=2}^{T} W_{hh} \cdot \text{diag}(\tanh'(z_t))$$

If $||W_{hh}|| < 1$: gradients → 0 (vanish) — **can't learn long dependencies**  
If $||W_{hh}|| > 1$: gradients → ∞ (explode) — **training diverges**

In [ ]:
import matplotlib.pyplot as plt

# Demonstrate vanishing/exploding gradients
np.random.seed(42)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, w_scale, title in [(axes[0], 0.5, 'Vanishing (||W|| < 1)'),
                            (axes[1], 1.5, 'Exploding (||W|| > 1)')]:
    W = np.random.randn(16, 16) * w_scale / 4
    gradient_norms = []
    g = np.ones(16)
    
    for t in range(50):
        g = W.T @ g  # Simplified gradient flow backward through time
        gradient_norms.append(np.linalg.norm(g))
    
    ax.plot(gradient_norms, lw=2, color='red' if w_scale > 1 else 'blue')
    ax.set_xlabel('Time Steps Backward')
    ax.set_ylabel('Gradient Norm')
    ax.set_title(title)
    ax.set_yscale('log')

plt.suptitle('Why Vanilla RNNs Fail: Vanishing vs Exploding Gradients', fontweight='bold')
plt.tight_layout()
plt.savefig('rnn_gradients.png', dpi=100)
plt.show()

print('After 50 time steps, gradient signal is either 0 or infinity.')
print('This means a vanilla RNN CANNOT learn "The cat, which I saw yesterday, sat."')
print('It cannot connect "cat" (step 1) to "sat" (step 7) — the gradient vanished.')

---
## 3 · LSTM: Long Short-Term Memory

LSTMs fix vanishing gradients with **gating mechanisms** — learnable valves that control information flow.

| Gate | Formula | Purpose |
|------|---------|--------|
| **Forget** ($f_t$) | $\sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$ | What to DELETE from memory |
| **Input** ($i_t$) | $\sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$ | What to WRITE to memory |
| **Output** ($o_t$) | $\sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$ | What to READ from memory |

The **cell state** ($c_t$) acts as a conveyor belt that carries information across time steps with minimal gradient degradation:

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$
$$h_t = o_t \odot \tanh(c_t)$$

In [ ]:
import torch
import torch.nn as nn

class LSTMFromScratch(nn.Module):
    """LSTM cell implemented from scratch."""
    
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # All four gates combined into one matrix for efficiency
        # (forget, input, candidate, output) = 4 * hidden_dim
        self.gates = nn.Linear(input_dim + hidden_dim, 4 * hidden_dim)
    
    def forward(self, x_sequence):
        """Process a sequence through the LSTM."""
        batch_size, seq_len, _ = x_sequence.shape
        
        h = torch.zeros(batch_size, self.hidden_dim, device=x_sequence.device)
        c = torch.zeros(batch_size, self.hidden_dim, device=x_sequence.device)
        
        outputs = []
        
        for t in range(seq_len):
            x_t = x_sequence[:, t, :]  # Current input
            combined = torch.cat([x_t, h], dim=1)
            gate_values = self.gates(combined)
            
            # Split into 4 gates
            f, i, c_tilde, o = gate_values.chunk(4, dim=1)
            
            f = torch.sigmoid(f)         # Forget gate: what to delete
            i = torch.sigmoid(i)         # Input gate: what to write
            c_tilde = torch.tanh(c_tilde)  # Candidate values to write
            o = torch.sigmoid(o)         # Output gate: what to read
            
            # Update cell state (the conveyor belt)
            c = f * c + i * c_tilde      # Forget old + write new
            
            # Compute output hidden state
            h = o * torch.tanh(c)        # Read from cell
            
            outputs.append(h.unsqueeze(1))
        
        return torch.cat(outputs, dim=1), (h, c)

# Compare: our LSTM vs PyTorch's built-in
input_dim, hidden_dim = 8, 16
ours = LSTMFromScratch(input_dim, hidden_dim)
official = nn.LSTM(input_dim, hidden_dim, batch_first=True)

x = torch.randn(4, 10, input_dim)  # Batch=4, seq=10, features=8

our_out, (our_h, our_c) = ours(x)
off_out, (off_h, off_c) = official(x)

print(f'Our LSTM output shape: {our_out.shape}')    # (4, 10, 16)
print(f'PyTorch LSTM output:   {off_out.shape}')     # (4, 10, 16)
print(f'Shapes match: {our_out.shape == off_out.shape} ✓')

print(f'\nOur params:     {sum(p.numel() for p in ours.parameters()):,}')
print(f'PyTorch params: {sum(p.numel() for p in official.parameters()):,}')
print('(PyTorch has separate input/recurrent weight matrices = more params)')

---
## 4 · Practical: Sentiment Classification with LSTM

In [ ]:
import torch
import torch.nn as nn

class SentimentLSTM(nn.Module):
    """Production-style LSTM for text classification."""
    
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                          batch_first=True, dropout=0.3, bidirectional=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),  # *2 for bidirectional
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        # Use the last time step's output (contains full sequence info)
        last_output = lstm_out[:, -1, :]
        return self.classifier(last_output)

model = SentimentLSTM(vocab_size=10000, embed_dim=128, hidden_dim=64, num_classes=3)
total_params = sum(p.numel() for p in model.parameters())

# Fake batch of tokenized reviews (padded to length 50)
x = torch.randint(0, 10000, (8, 50))
output = model(x)

print(f'SentimentLSTM')
print(f'  Parameters:  {total_params:,}')
print(f'  Input shape:  {x.shape} (batch=8, seq_len=50, token IDs)')
print(f'  Output shape: {output.shape} (batch=8, 3 classes: pos/neu/neg)')
print(f'  Bidirectional: ✓ (reads forward AND backward)')
print(f'\n  Compare to BERT: ~110M params vs our {total_params/1e6:.1f}M')
print(f'  BERT is 10x larger but captures much richer context')

---
## 5 · RNN vs Transformer — When to Use What

### 🎓 DEEP QUESTION ANSWERED

**Q:** *If transformers are faster AND better, why do some systems still use LSTMs?*

**A:**

| Factor | LSTM | Transformer |
|--------|------|-------------|
| Model size | ~1-10M params | ~100M - 1T params |
| Inference speed (short seq) | Very fast | Slower (attention overhead) |
| Memory (inference) | O(1) per step | O(n) for KV-cache |
| Long dependencies (>200 tokens) | ❌ Fails | ✅ Excels |
| Edge deployment | ✅ Fits on microcontroller | ❌ Needs GPU |
| Training parallelism | ❌ Sequential | ✅ Fully parallel |

**Production Rule of Thumb:** Use transformers for NLP. Use LSTMs for time-series on edge devices. If in doubt, use a transformer.

---

## ✅ Knowledge Check

### Q1: Gate intuition
In the LSTM forget gate, what happens when $f_t ≈ 0$ for all dimensions?

<details><summary>👉 Answer</summary>

The cell state is completely erased ($c_t = 0 \cdot c_{t-1} + ...$). The LSTM forgets ALL previous information. This happens at sentence boundaries or topic changes. The model learns when to reset its memory.
</details>

### Q2: Bidirectional
Why can't you use a bidirectional LSTM for language generation (like GPT)?

<details><summary>👉 Answer</summary>

Bidirectional LSTMs read the entire sequence (forward AND backward) before producing output. But in generation, future tokens don't exist yet — you're generating them! You can only use unidirectional (left-to-right) for generation. Same reason GPT uses causal masking instead of BERT's bidirectional attention.
</details>

### Q3: Parameter count
An LSTM with input_dim=256 and hidden_dim=512 has how many parameters in ONE cell? (Hint: 4 gates, each with input and recurrent weights + biases)

<details><summary>👉 Answer</summary>

Each gate: $W_{xh}$ (256×512) + $W_{hh}$ (512×512) + bias (512) = 131,072 + 262,144 + 512 = 393,728  
Four gates: 4 × 393,728 = 1,574,912 parameters (~1.5M)
</details>

---

## 🔨 Practical Practice

### Exercise 1: Gradient Flow Comparison
Run a gradient flow experiment: pass a 100-step sequence through (a) Vanilla RNN and (b) LSTM. Compare the gradient norms at step 0 w.r.t. the loss at step 100.

### Exercise 2: Character-Level Language Model
Build a character-level LSTM that trains on Shakespeare text and generates new text one character at a time.

### Exercise 3: GRU Implementation
Implement a GRU cell from scratch. GRU uses only 2 gates (reset + update) instead of LSTM's 3. Compare parameter counts.

---

**Next →** NLP 04: Sequence-to-Sequence & the Attention Mechanism